# Adım 2: Kafka ile Streaming Veri Üretimi

Bu notebook NYC Taxi verisini Kafka topic'ine JSON formatında ileten **Python Kafka Producer** bileşenini açıklar ve test eder.

| Parametre | Değer |
|-----------|-------|
| Veri kaynağı | `data/raw/yellow_tripdata_2023-01.parquet` |
| Kafka topic | `yellow_taxi_trips` |
| Mesaj formatı | JSON (timestamp, alanlar, key=PULocationID) |
| Gönderim hızı | Ayarlanabilir (batch + sleep) |

## 1. Gerekli Kütüphaneler

In [ ]:
import subprocess, sys

required = ["kafka-python==2.0.2", "pyarrow==15.0.2", "numpy==1.26.4"]
for pkg in required:
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", pkg, "-q"],
        capture_output=True, text=True
    )
    status = "✅" if result.returncode == 0 else "❌"
    print(f"{status} {pkg}")

## 2. Konfigürasyon

Tüm parametreler ortam değişkenleriyle veya aşağıdan ayarlanabilir:

In [ ]:
import os
from pathlib import Path

PROJECT_ROOT = Path(os.getcwd()).parent

CONFIG = {
    "KAFKA_BOOTSTRAP_SERVERS": os.getenv("KAFKA_BOOTSTRAP_SERVERS", "localhost:29092"),
    "KAFKA_TOPIC":             os.getenv("KAFKA_TOPIC",             "yellow_taxi_trips"),
    "PARQUET_PATH":            os.getenv("RAW_TAXI_PARQUET",        str(PROJECT_ROOT / "data/raw/yellow_tripdata_2023-01.parquet")),
    "MAX_ROWS":                int(os.getenv("PRODUCER_MAX_ROWS",   "500")),
    "BATCH_SIZE":              int(os.getenv("PRODUCER_BATCH_SIZE", "50")),
    "SLEEP_SECONDS":           float(os.getenv("PRODUCER_SLEEP_SECONDS", "0.2")),
    "KEY_FIELD":               os.getenv("PRODUCER_KEY_FIELD",      "PULocationID"),
}

print("=== Producer Konfigürasyonu ===")
for k, v in CONFIG.items():
    print(f"  {k:<30} = {v}")

## 3. Parquet Veri Setinin İncelenmesi

In [ ]:
import pyarrow.parquet as pq

parquet_path = CONFIG["PARQUET_PATH"]

if not Path(parquet_path).exists():
    print(f"❌ Dosya bulunamadı: {parquet_path}")
    print("   data/raw/ klasörüne yellow_tripdata_2023-01.parquet dosyasını indirin.")
else:
    pf = pq.ParquetFile(parquet_path)
    meta = pf.metadata
    schema = pf.schema_arrow

    print(f"✅ Dosya bulundu: {parquet_path}")
    print(f"   Toplam satır  : {meta.num_rows:,}")
    print(f"   Row group sayısı: {meta.num_row_groups}")
    print(f"   Sütun sayısı  : {meta.num_columns}")
    print(f"\nŞema:")
    for field in schema:
        print(f"  {field.name:<35} {field.type}")

## 4. İlk Kayıtların Önizlemesi

In [ ]:
import json, math
from datetime import datetime, date
from decimal import Decimal
import numpy as np

def normalize_value(v):
    if v is None: return None
    if isinstance(v, float) and math.isnan(v): return None
    if isinstance(v, (datetime, date)): return v.isoformat()
    if isinstance(v, Decimal): return float(v)
    if isinstance(v, np.integer): return int(v)
    if isinstance(v, np.floating): return None if np.isnan(v) else float(v)
    if isinstance(v, bytes): return v.decode("utf-8", errors="replace")
    return v

def normalize_record(rec):
    return {k: normalize_value(vv) for k, vv in rec.items()}

if Path(parquet_path).exists():
    batch = next(pq.ParquetFile(parquet_path).iter_batches(batch_size=3))
    ornek = [normalize_record(r) for r in batch.to_pylist()]
    for i, rec in enumerate(ornek, 1):
        print(f"--- Kayıt {i} ---")
        print(json.dumps(rec, indent=2, ensure_ascii=False))
        print()

## 5. JSON Mesaj Yapısının Doğrulanması

Her mesajda `timestamp`, `PULocationID` (key), ve tüm trip alanları bulunur:

In [ ]:
from datetime import timezone

def enrich_record(rec: dict) -> dict:
    """Producer'ın gönderdiği gerçek mesaj formatı."""
    rec["_ingested_at"] = datetime.now(timezone.utc).isoformat()
    return rec

if Path(parquet_path).exists():
    ornek_mesaj = enrich_record(ornek[0])
    payload_str = json.dumps(ornek_mesaj, ensure_ascii=False)
    key = str(ornek_mesaj.get(CONFIG["KEY_FIELD"], "unknown"))

    print(f"Mesaj Key   : {key}")
    print(f"Payload boyutu: {len(payload_str.encode())} byte")
    print(f"\nAlan sayısı : {len(ornek_mesaj)}")
    zorunlu = ["tpep_pickup_datetime", "tpep_dropoff_datetime",
               "passenger_count", "trip_distance",
               "PULocationID", "DOLocationID",
               "fare_amount", "total_amount", "_ingested_at"]
    print("\nZorunlu Alan Kontrolü:")
    for alan in zorunlu:
        deger = ornek_mesaj.get(alan, "EKSİK")
        icon = "✅" if alan in ornek_mesaj else "❌"
        print(f"  {icon} {alan:<35} = {deger}")

## 6. Kafka Bağlantı Testi

In [ ]:
from kafka import KafkaProducer, KafkaAdminClient
from kafka.errors import KafkaError

bootstrap = CONFIG["KAFKA_BOOTSTRAP_SERVERS"]

try:
    admin = KafkaAdminClient(bootstrap_servers=bootstrap, request_timeout_ms=5000)
    topics = admin.list_topics()
    admin.close()
    print(f"✅ Kafka broker'a bağlanıldı: {bootstrap}")
    print(f"   Mevcut topic'ler: {topics if topics else '(henüz yok)'}")
except Exception as e:
    print(f"❌ Kafka bağlantı hatası: {e}")
    print("   → Adım 1 notebook'unu çalıştırarak servisleri başlatın.")

## 7. Producer Fonksiyonlarının İncelenmesi

`producer/main.py` dosyasının temel fonksiyonları:

In [ ]:
producer_path = PROJECT_ROOT / "producer" / "main.py"
with open(producer_path, "r", encoding="utf-8") as f:
    kod = f.read()

# Sadece fonksiyon imzalarını göster
import re
fonksiyonlar = re.findall(r'^def (\w+\([^)]*\)).*', kod, re.MULTILINE)
print("producer/main.py — Fonksiyonlar:")
print("-" * 50)
for fn in fonksiyonlar:
    print(f"  def {fn}")

## 8. Ayarlanabilir Hız Parametreleri

Producer'ın gönderim hızı `BATCH_SIZE` ve `SLEEP_SECONDS` ile kontrol edilir:

In [ ]:
senaryolar = [
    {"ad": "Yavaş  (test)",   "batch": 10,  "sleep": 1.0},
    {"ad": "Normal (varsayılan)", "batch": 50,  "sleep": 0.2},
    {"ad": "Hızlı  (yük testi)", "batch": 500, "sleep": 0.0},
]

print(f"{'Senaryo':<25} {'Batch':<8} {'Sleep':<8} {'Msg/sn (tahmini)'}")
print("-" * 60)
for s in senaryolar:
    hiz = s["batch"] / max(s["sleep"], 0.001)
    print(f"{s['ad']:<25} {s['batch']:<8} {s['sleep']:<8} ~{hiz:.0f} msg/sn")

## 9. Canlı Producer Testi (Dry-Run)

Kafka'ya **gerçekten göndermeden** mesajların serialize edildiğini doğrular:

In [ ]:
import time

def json_default(v):
    if isinstance(v, (datetime, date)): return v.isoformat()
    if isinstance(v, Decimal): return float(v)
    if isinstance(v, np.integer): return int(v)
    if isinstance(v, np.floating): return None if np.isnan(v) else float(v)
    if isinstance(v, float) and math.isnan(v): return None
    return str(v)

if Path(parquet_path).exists():
    MAX_ROWS   = 100
    BATCH_SIZE = 25
    topic      = CONFIG["KAFKA_TOPIC"]
    key_field  = CONFIG["KEY_FIELD"]

    rows_seen, batch = 0, []
    start = time.time()

    for rb in pq.ParquetFile(parquet_path).iter_batches(batch_size=BATCH_SIZE):
        for rec in rb.to_pylist():
            if rows_seen >= MAX_ROWS:
                break
            batch.append(normalize_record(rec))
            rows_seen += 1
        if rows_seen >= MAX_ROWS:
            break
        if len(batch) >= BATCH_SIZE:
            for r in batch:
                json.dumps(r, default=json_default).encode("utf-8")
            key = str(batch[0].get(key_field, "unknown"))
            print(f"[DRY-RUN] topic={topic} batch={len(batch)} key_sample={key}")
            batch = []

    elapsed = time.time() - start
    print(f"\n✅ Dry-run tamamlandı: {rows_seen} kayıt, {elapsed:.2f}s")

## 10. Docker ile Producer'ı Başlatma

Gerçek Kafka gönderimi için Docker Compose'daki `producer` servisini kullanın:

In [ ]:
import subprocess, os
os.chdir(PROJECT_ROOT)

# Producer'ı başlat (pipeline profili ile)
cmd = "docker compose --profile pipeline up producer -d"
print(f"Komut: {cmd}\n")

result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
print(result.stdout)
if result.stderr:
    print(result.stderr)

## 11. Producer Loglarının Takibi

In [ ]:
# Son 30 satır log
result = subprocess.run(
    "docker logs nyc-taxi-producer --tail 30",
    shell=True, capture_output=True, text=True
)
print(result.stdout or result.stderr or "(henüz log yok)")

## 12. Kafka Topic Doğrulaması

Topic oluşturuldu mu ve kaç mesaj var?

In [ ]:
topic = CONFIG["KAFKA_TOPIC"]

print("=== Topic Listesi ===")
r1 = subprocess.run(
    "docker exec nyc-taxi-kafka kafka-topics --bootstrap-server localhost:9092 --list",
    shell=True, capture_output=True, text=True
)
print(r1.stdout or "(henüz topic yok)")

print(f"\n=== '{topic}' Topic Detayı ===")
r2 = subprocess.run(
    f"docker exec nyc-taxi-kafka kafka-topics --bootstrap-server localhost:9092 "
    f"--describe --topic {topic}",
    shell=True, capture_output=True, text=True
)
print(r2.stdout or r2.stderr or "(topic henüz oluşturulmadı)")

## 13. Kafka Consumer ile Mesaj Örnekleme

Topic'e ulaşan ilk birkaç mesajı okuyalım:

In [ ]:
from kafka import KafkaConsumer

OKUMA_LIMITI = 5

try:
    consumer = KafkaConsumer(
        CONFIG["KAFKA_TOPIC"],
        bootstrap_servers=CONFIG["KAFKA_BOOTSTRAP_SERVERS"],
        auto_offset_reset="earliest",
        consumer_timeout_ms=5000,
        value_deserializer=lambda b: json.loads(b.decode("utf-8")),
    )
    okunan = 0
    for msg in consumer:
        print(f"\n--- Mesaj {okunan+1} ---")
        print(f"  Partition : {msg.partition}")
        print(f"  Offset    : {msg.offset}")
        print(f"  Key       : {msg.key}")
        payload = msg.value
        print(f"  Alan sayısı: {len(payload)}")
        for alan in ["tpep_pickup_datetime", "PULocationID", "DOLocationID", "total_amount"]:
            print(f"  {alan:<30} = {payload.get(alan)}")
        okunan += 1
        if okunan >= OKUMA_LIMITI:
            break
    consumer.close()
    print(f"\n✅ Toplam {okunan} mesaj okundu.")
except Exception as e:
    print(f"❌ Hata: {e}")

## 14. Gönderim İstatistikleri

In [ ]:
log_output = subprocess.run(
    "docker logs nyc-taxi-producer 2>&1",
    shell=True, capture_output=True, text=True
).stdout

published_lines = [l for l in log_output.splitlines() if "Published" in l]
finished_lines  = [l for l in log_output.splitlines() if "Finished"  in l]

if published_lines:
    print("Son gönderim logları:")
    for line in published_lines[-5:]:
        print(f"  {line}")
if finished_lines:
    print("\nTamamlanma mesajı:")
    print(f"  {finished_lines[-1]}")
if not published_lines and not finished_lines:
    print("(Henüz gönderim logu yok — producer çalışmıyor olabilir)")

## 15. Özet & Teslim Edilecekler

| # | Teslim | Durum |
|---|--------|-------|
| 1 | `producer/main.py` — Parquet okuma, JSON serialize, Kafka gönderim | ✅ |
| 2 | Her mesajda timestamp + tüm trip alanları | ✅ |
| 3 | Ayarlanabilir hız (`BATCH_SIZE`, `SLEEP_SECONDS`) | ✅ |
| 4 | Producer logları ile mesaj takibi | ✅ (Bölüm 11-14) |

### Ortam Değişkenleri ile Hız Ayarı

```bash
# Hızlı gönderim
PRODUCER_BATCH_SIZE=500 PRODUCER_SLEEP_SECONDS=0.05 docker compose --profile pipeline up producer

# Yavaş / test modu
PRODUCER_BATCH_SIZE=10 PRODUCER_SLEEP_SECONDS=1.0 docker compose --profile pipeline up producer
```

---
> **Sonraki adım →** Adım 3: Apache Spark ile stream işleme